In [ ]:
%reset -f 
# resetting stored variables in case there's something weird cached
from build123d import *
from ocp_vscode import *
import cadquery as cq
import time
import math
from library.tools import *
import sys
from dataclasses import dataclass, field
import bd_warehouse.thread, bd_warehouse.fastener 
from pathlib import Path
from sympy import false
from casadi import diag
from build123d.topology.composite import Part
import path
import yaml

ALL UNITS IN MM
DON'T @ ME

In [740]:
%reload_ext ocp_vscode
%load_ext autoreload
%autoreload 
print(f"Python executable: {sys.executable}")
print(f"OCP-vscode location: {sys.modules.get('ocp_vscode', 'Not found')}")
reset_show()
startTime = time.perf_counter()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python executable: c:\Users\Kaoti\parthenon\.venv\Scripts\python.exe
OCP-vscode location: <module 'ocp_vscode' from 'c:\\Users\\Kaoti\\parthenon\\.venv\\Lib\\site-packages\\ocp_vscode\\__init__.py'>


In [741]:
# INPUTS GO HERE
# Neurotrophic Geometry/Solids Pathfinder/Originator/Generator/Optimizer? NOPE
# Neurotrophic Rhesus Optimized 
# NUTS
# Neurotrophic United/unionizing/unironical/underloft Topology Solution?
# ROOTS: Reconfigurable Originator of Translational Solids

# Chamber identity
chamberIdentity = "GoliathPosterior" # Other possibilities: GoliathAnterior, MalachiRight, MalachiLeft. More TBA
softwareVersion = 1.0

# Just for the moment we are presuming a universal x value, so...
# at xValue, yOffset = (2,3) we start getting issues with geometry overlapping in funny ways. Make nubs smaller?
# Same as above at (2,-4)
xValue = 1.75 # Domain: -5 to +5 mm
yOffset = 0 # Range: -3, -2, -1, 0, 1, 2, 3...
iteratorOffset = 5 #5 works, 6 works, 4 does not

"""
Original y values, 1-4: 7.7, 2.475, -1.775, -7
"""

penetration1 = (xValue, 4.75)
penetration2 = (xValue, -0.25)
penetration3 = (xValue, -4.5)
penetration4 = (xValue, -9.75)
"""
# Testing: 
# Coordinate sites / Penetration center points - these will be inputs which generate whole structure
penetration1 = (xValue, (iteratorOffset * 1.5)+ yOffset)
penetration2 = (xValue, (iteratorOffset * 0.5) + yOffset)
penetration3 = (xValue, (-iteratorOffset * 0.5) + yOffset)
penetration4 = (xValue, (-iteratorOffset * 1.5) + yOffset)
"""
points2D = (penetration1, penetration2, penetration3, penetration4)

# Diagnostic mode yes or no? If y, then we should set it up to have Show() commands for diagnostic purposes that turn on with a time.sleep() so we can do analysis and show off what's happening. 
diagnosticMode = 3 # user input field eventually. 0 is no, 1 is full diagnostics, 2 is timefield reporting for subfield functions only. 
smallDiagnosticTime = 0.1
largeDiagnosticTime = 0.5

# Do you want to save this file? 0 is no, 1 is yes. Needs a file name; this will be used as a suffix. 
fileName = f"ParametricGuideTubeFrameV{softwareVersion}"
saveyn = 1

In [742]:
# Below are some parameters which govern the GT shaft dimensions. Easy permutation layer for later designs if needed.
shaftHeight = 3.85 # This is a rough estimation based on Anna's onshape
shaftDiameter = 4
innerShaftDiameter = 0.57
zOffset = 2

# These should probably be divined algorithmically but right now they're just the victims of trial and error on my part
handPickedIndicies = [1,5,9,13]

In [743]:
# Site-based data storage 

sites = []
planeHeightIndicator = 0

# Initialize the class and contents
@dataclass
class Site:
    shaft: Part
    plane: Plane
    nubs: list
    arms: list = field(default_factory = list)
    outerDiameter: object = None
    innerDiameter: object = None
    combined: object = None


In [744]:
# Calling chamberCylinder to construct the basisCylinder
basisCylinder = chamberCylinder(chamberIdentity,diagnosticMode)
# note: 10% of time

In [745]:
# Calling meshPlanes to get our starting extrusion planes, ending extrusion solids, and 3D coordinates for each site
points3D, bottomSurfaceSolids, startingOffsetPlanes = meshPlanes(chamberIdentity=chamberIdentity, points2D=points2D, ZOffset = zOffset, shaftHeight = shaftHeight, shaftDiameter = shaftDiameter, diagnosticMode = diagnosticMode, largeDiagnosticTime = 0.5, smallDiagnosticTime = 0.10)

In [746]:
# Calling shaftConstructor to construct shafts at each site
shaftListWithThroughHolesFilletedTwice = shaftConstructor(startingOffsetPlanes, bottomSurfaceSolids, innerShaftDiameter = innerShaftDiameter, shaftDiameter = shaftDiameter, diagnosticMode = diagnosticMode, largeDiagnosticTime = largeDiagnosticTime, smallDiagnosticTime = smallDiagnosticTime)

In [747]:
# Constructing nubs for each site
nubsList, nubTemplatesFlattened = nubConstructor(startingOffsetPlanes, diagnosticMode = diagnosticMode, largeDiagnosticTime = largeDiagnosticTime, smallDiagnosticTime = smallDiagnosticTime)

In [748]:
# Constructing intermediate geometry with lofts between sites
prunedParts, lofts, overlapSolids = loftConstructor(nubsList, nubTemplatesFlattened, startingOffsetPlanes)

In [749]:
# Generating positional arms to connect nubs and basis cylinder
outputArmStream, testArms = armStreamConstructor(startingOffsetPlanes,nubsList,handPickedIndicies)

In [750]:
# Constructing initial sites, storing previously gathered information in them
sites = siteConstructor(Site, startingOffsetPlanes,shaftListWithThroughHolesFilletedTwice,nubsList,prunedParts,outputArmStream)

In [751]:
# Construct guide tube frame from all the parts we've just made
guideTubeFrame = guideTubeFrameConstructor(sites, basisCylinder, nubsList, prunedParts, lofts, overlapSolids)

In [752]:
# Saving file as STL if saveyn = 1, and printing the file name
finalName = saveMe(points2D,startTime,guideTubeFrame,saveyn,fileName,chamberIdentity)

Saved as: GoliathPosterior-(1.75, 4.75)-(1.75, -0.25)-(1.75, -4.5)-(1.75, -9.75)-ParametricGuideTubeFrameV1.0.stl
